# 💬 Sentiment Lab — Analyse de sentiments multi-modèles

**TF-IDF + LogReg/SVM · MLP · CNN 1D (Kim)** avec une interface interactive (Gradio).

### Comment l'utiliser (3 étapes)
1. **Exécution → Tout exécuter** (ou `Ctrl+F9`). Les cellules de code source s'exécutent en quelques secondes.
2. À la dernière cellule, l'interface s'affiche **directement dans le notebook** (et un lien public `*.gradio.live` est généré pour l'ouvrir en plein écran).
3. Dans l'interface : **Données & EDA → Entraînement → Résultats → Démo live**.

> 💡 Pour accélérer le CNN : *Exécution → Modifier le type d'exécution → GPU*.

## 1️⃣ Installation

In [ ]:
!pip install -q "gradio>=5,<6" datasets tabulate
import nltk
nltk.download("stopwords", quiet=True); nltk.download("wordnet", quiet=True)
print("✅ Dépendances prêtes")

## 2️⃣ Code source du projet
Les cellules suivantes écrivent les fichiers du projet dans `/content/sentiment-multimodel/` (structure modulaire : `src/` + pipeline + interface).

In [ ]:
import os
os.makedirs("/content/sentiment-multimodel/src", exist_ok=True)

In [ ]:
%%writefile /content/sentiment-multimodel/src/__init__.py
# package

In [ ]:
%%writefile /content/sentiment-multimodel/src/config.py
"""Configuration centrale du projet."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
FIG_DIR = ROOT / "reports" / "figures"
REPORT_DIR = ROOT / "reports"

for d in (DATA_DIR, MODELS_DIR, FIG_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
LABELS = ["négatif", "neutre", "positif"]  # 0, 1, 2 (convention tweet_eval)
NUM_CLASSES = len(LABELS)

# Dataset Hugging Face (3 classes). Remplaçable par un CSV local (colonnes: text,label)
HF_DATASET = ("cardiffnlp/tweet_eval", "sentiment")  # ancien nom court "tweet_eval" refusé par huggingface_hub récent
HF_DATASET_FALLBACK = ("tweet_eval", "sentiment")
LOCAL_CSV = DATA_DIR / "reviews.csv"

# Séquences / vocabulaire
MAX_LEN = 50
VOCAB_SIZE = 20_000
EMB_DIM = 128

# TF-IDF
TFIDF_MAX_FEATURES = 20_000
TFIDF_NGRAMS = (1, 2)

# Entraînement NN
EPOCHS = 15
BATCH_SIZE = 64

In [ ]:
%%writefile /content/sentiment-multimodel/src/data.py
"""Chargement des données : Hugging Face (tweet_eval) ou CSV local."""
import pandas as pd
from sklearn.model_selection import train_test_split

from src import config


def load_data():
    """Retourne (train, val, test) : DataFrames avec colonnes `text` et `label`.

    - Si data/reviews.csv existe (colonnes text,label), il est utilisé et découpé 70/15/15.
    - Sinon, le dataset Hugging Face `tweet_eval/sentiment` est téléchargé.
    """
    if config.LOCAL_CSV.exists():
        df = pd.read_csv(config.LOCAL_CSV)[["text", "label"]].dropna()
        train, tmp = train_test_split(
            df, test_size=0.30, stratify=df["label"], random_state=config.SEED
        )
        val, test = train_test_split(
            tmp, test_size=0.50, stratify=tmp["label"], random_state=config.SEED
        )
    else:
        from datasets import load_dataset

        try:
            ds = load_dataset(*config.HF_DATASET)
        except Exception as e:  # nom du dépôt changé / réseau : on tente l'ancien nom
            print(f"Échec avec {config.HF_DATASET[0]} ({type(e).__name__}), essai de secours...")
            try:
                ds = load_dataset(*config.HF_DATASET_FALLBACK)
            except Exception:
                raise RuntimeError(
                    "Impossible de télécharger le dataset Hugging Face. Vérifie la connexion "
                    "Internet de Colab, ou utilise l'option « Mon fichier CSV » "
                    "(colonnes text,label)."
                ) from e
        train, val, test = (
            ds[s].to_pandas()[["text", "label"]] for s in ("train", "validation", "test")
        )
    return (d.reset_index(drop=True) for d in (train, val, test))

In [ ]:
%%writefile /content/sentiment-multimodel/src/preprocess.py
"""Prétraitement NLP : nettoyage, stopwords, lemmatisation, tokenisation, padding."""
import re

import nltk
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

from src import config


def _ensure_nltk():
    for pkg, path in [("stopwords", "corpora/stopwords"), ("wordnet", "corpora/wordnet")]:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)


_ensure_nltk()

# On garde les négations : elles portent le sentiment ("not good" != "good")
_NEGATIONS = {"no", "nor", "not", "don", "didn", "doesn", "isn", "wasn", "aren",
              "weren", "won", "wouldn", "couldn", "shouldn", "hasn", "haven", "hadn"}
STOPWORDS = set(stopwords.words("english")) - _NEGATIONS
_lemmatizer = WordNetLemmatizer()

_URL = re.compile(r"https?://\S+|www\.\S+")
_MENTION = re.compile(r"@\w+")
_NON_ALPHA = re.compile(r"[^a-z\s]")
_SPACES = re.compile(r"\s+")


def clean_text(text: str, remove_stopwords: bool = True, lemmatize: bool = True) -> str:
    """minuscules → suppression URL/mentions/ponctuation → stopwords → lemmatisation."""
    text = text.lower()
    text = _URL.sub(" ", text)
    text = _MENTION.sub(" ", text)
    text = text.replace("#", " ")
    text = _NON_ALPHA.sub(" ", text)
    tokens = _SPACES.sub(" ", text).strip().split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    if lemmatize:
        tokens = [_lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)


def clean_series(series):
    return series.astype(str).map(clean_text)


def fit_tokenizer(train_texts) -> Tokenizer:
    tok = Tokenizer(num_words=config.VOCAB_SIZE, oov_token="<OOV>")
    tok.fit_on_texts(train_texts)
    return tok


def to_padded(tok: Tokenizer, texts) -> np.ndarray:
    seqs = tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=config.MAX_LEN, padding="post", truncating="post")

In [ ]:
%%writefile /content/sentiment-multimodel/src/eda.py
"""Analyse exploratoire (EDA) : classes, longueurs, mots fréquents."""
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import config

sns.set_theme(style="whitegrid")


def run_eda(train: pd.DataFrame, clean_texts: pd.Series) -> None:
    df = train.copy()
    df["sentiment"] = df["label"].map(dict(enumerate(config.LABELS)))
    df["n_words"] = df["text"].str.split().str.len()
    df["n_chars"] = df["text"].str.len()

    # 1) Distribution des classes
    fig, ax = plt.subplots(figsize=(5, 4))
    order = config.LABELS
    sns.countplot(data=df, x="sentiment", order=order, ax=ax)
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom")
    ax.set_title("Distribution des classes (train)")
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / "eda_class_distribution.png", dpi=150)
    plt.close(fig)

    # 2) Longueur des textes
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.histplot(df["n_words"], bins=40, ax=axes[0])
    axes[0].set_title("Nombre de mots par texte")
    sns.boxplot(data=df, x="sentiment", y="n_words", order=order, ax=axes[1])
    axes[1].set_title("Longueur par classe")
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / "eda_text_length.png", dpi=150)
    plt.close(fig)

    # 3) Mots les plus fréquents par classe (texte nettoyé)
    fig, axes = plt.subplots(1, config.NUM_CLASSES, figsize=(15, 5))
    for i, name in enumerate(order):
        words = Counter(" ".join(clean_texts[df["label"] == i]).split()).most_common(15)
        w, c = zip(*words)
        sns.barplot(x=list(c), y=list(w), ax=axes[i])
        axes[i].set_title(f"Top 15 mots — {name}")
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / "eda_top_words.png", dpi=150)
    plt.close(fig)

    stats = df.groupby("sentiment")[["n_words", "n_chars"]].describe().round(1)
    stats.to_csv(config.REPORT_DIR / "eda_stats.csv")
    print("EDA terminée → reports/figures/")

In [ ]:
%%writefile /content/sentiment-multimodel/src/baseline.py
"""Baseline classique : TF-IDF + Logistic Regression / SVM linéaire (la référence à battre)."""
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from src import config


def make_tfidf() -> TfidfVectorizer:
    return TfidfVectorizer(
        max_features=config.TFIDF_MAX_FEATURES,
        ngram_range=config.TFIDF_NGRAMS,
        sublinear_tf=True,
        min_df=2,
    )


def train_baselines(X_train, y_train):
    """Retourne un dict {nom: pipeline entraîné}."""
    models = {
        "TF-IDF + LogReg": Pipeline([
            ("tfidf", make_tfidf()),
            ("clf", LogisticRegression(max_iter=1000, C=2.0, random_state=config.SEED)),
        ]),
        "TF-IDF + LinearSVM": Pipeline([
            ("tfidf", make_tfidf()),
            ("clf", LinearSVC(C=0.5, random_state=config.SEED)),
        ]),
    }
    for name, pipe in models.items():
        print(f"Entraînement {name} ...")
        pipe.fit(X_train, y_train)
    return models


def save(pipe, name: str):
    joblib.dump(pipe, config.MODELS_DIR / f"{name}.joblib")

In [ ]:
%%writefile /content/sentiment-multimodel/src/mlp.py
"""MLP sur vecteurs TF-IDF : Dense → Dropout → Dense → Dropout → Softmax."""
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

from src import config


class SparseBatches(keras.utils.Sequence):
    """Densifie la matrice creuse TF-IDF batch par batch (évite d'exploser la RAM)."""

    def __init__(self, X, y=None, batch_size=config.BATCH_SIZE, shuffle=False):
        super().__init__()
        self.X, self.y, self.bs, self.shuffle = X, y, batch_size, shuffle
        self.idx = np.arange(X.shape[0])
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.idx) / self.bs))

    def __getitem__(self, i):
        b = self.idx[i * self.bs:(i + 1) * self.bs]
        xb = self.X[b].toarray().astype("float32")
        return (xb, self.y[b]) if self.y is not None else xb

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.idx)


def build_mlp(input_dim: int) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(config.NUM_CLASSES, activation="softmax"),
    ], name="mlp_tfidf")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def train_mlp(Xtr, ytr, Xva, yva):
    model = build_mlp(Xtr.shape[1])
    cb = [keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True,
                                        monitor="val_loss")]
    history = model.fit(
        SparseBatches(Xtr, ytr, shuffle=True),
        validation_data=SparseBatches(Xva, yva),
        epochs=config.EPOCHS, callbacks=cb, verbose=2,
    )
    return model, history


def predict_proba(model, X):
    return model.predict(SparseBatches(X), verbose=0)

In [ ]:
%%writefile /content/sentiment-multimodel/src/cnn.py
"""CNN 1D pour texte, inspiré de Kim (2014) « Convolutional Neural Networks for Sentence Classification ».

Embedding → [Conv1D(k=3,4,5) → GlobalMaxPooling] en parallèle → Concat → Dropout → Dense → Softmax
"""
from tensorflow import keras
from tensorflow.keras import layers

from src import config


def build_kim_cnn(vocab_size=config.VOCAB_SIZE, emb_dim=config.EMB_DIM,
                  max_len=config.MAX_LEN, kernel_sizes=(3, 4, 5), n_filters=100,
                  embedding_matrix=None) -> keras.Model:
    inp = layers.Input(shape=(max_len,), name="tokens")
    if embedding_matrix is not None:  # ex. GloVe / Word2Vec pré-entraînés
        emb = layers.Embedding(vocab_size, embedding_matrix.shape[1],
                               weights=[embedding_matrix], trainable=True)(inp)
    else:
        emb = layers.Embedding(vocab_size, emb_dim)(inp)
    emb = layers.SpatialDropout1D(0.2)(emb)

    pooled = []
    for k in kernel_sizes:  # chaque noyau capte des n-grammes de taille k
        c = layers.Conv1D(n_filters, k, activation="relu")(emb)
        pooled.append(layers.GlobalMaxPooling1D()(c))
    x = layers.Concatenate()(pooled) if len(pooled) > 1 else pooled[0]
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(config.NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inp, out, name="kim_cnn")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def train_cnn(Xtr, ytr, Xva, yva):
    model = build_kim_cnn()
    cb = [keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True,
                                        monitor="val_loss")]
    history = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                        epochs=config.EPOCHS, batch_size=config.BATCH_SIZE,
                        callbacks=cb, verbose=2)
    return model, history

In [ ]:
%%writefile /content/sentiment-multimodel/src/evaluate.py
"""Évaluation comparative : métriques, matrices de confusion, courbes d'apprentissage."""
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)

from src import config

sns.set_theme(style="whitegrid")


def evaluate(name, y_true, y_pred) -> dict:
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro",
                                                  zero_division=0)
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=config.LABELS, digits=3))
    return {"modèle": name, "accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": p, "recall_macro": r, "f1_macro": f1}


def plot_confusion_matrices(preds: dict, y_true):
    n = len(preds)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4))
    axes = [axes] if n == 1 else axes
    for ax, (name, y_pred) in zip(axes, preds.items()):
        cm = confusion_matrix(y_true, y_pred, normalize="true")
        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=config.LABELS, yticklabels=config.LABELS)
        ax.set_title(name, fontsize=10)
        ax.set_xlabel("Prédit")
        ax.set_ylabel("Réel")
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / "confusion_matrices.png", dpi=150)
    plt.close(fig)


def plot_learning_curves(histories: dict):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, h in histories.items():
        ep = list(range(1, len(h.history["loss"]) + 1))
        axes[0].plot(ep, h.history["loss"], label=f"{name} train")
        axes[0].plot(ep, h.history["val_loss"], "--", label=f"{name} val")
        axes[1].plot(ep, h.history["accuracy"], label=f"{name} train")
        axes[1].plot(ep, h.history["val_accuracy"], "--", label=f"{name} val")
    axes[0].set(title="Loss", xlabel="Époque")
    axes[1].set(title="Accuracy", xlabel="Époque")
    for a in axes:
        a.xaxis.set_major_locator(MaxNLocator(integer=True))
        a.legend()
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / "learning_curves.png", dpi=150)
    plt.close(fig)


def save_comparison(rows: list):
    df = pd.DataFrame(rows).set_index("modèle").round(4)
    df.to_csv(config.REPORT_DIR / "results.csv")
    df.to_markdown(config.REPORT_DIR / "results.md")
    print("\n", df)

    ax = df[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].plot.bar(
        figsize=(10, 5), rot=15)
    ax.set_ylim(df.values.min() - 0.05, min(1.0, df.values.max() + 0.03))
    ax.set_title("Comparaison des modèles (jeu de test)")
    ax.set_xlabel("")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4, frameon=False)
    plt.savefig(config.FIG_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()

In [ ]:
%%writefile /content/sentiment-multimodel/train_all.py
"""Pipeline complet : EDA → prétraitement → baselines → MLP → CNN → comparaison.

Usage :  python train_all.py [--sample 5000]   (--sample pour un test rapide)
"""
import argparse
import json

import joblib
import numpy as np
import tensorflow as tf

from src import config
from src.baseline import make_tfidf, save, train_baselines
from src.cnn import train_cnn
from src.data import load_data
from src.eda import run_eda
from src.evaluate import (evaluate, plot_confusion_matrices, plot_learning_curves,
                          save_comparison)
from src.mlp import predict_proba, train_mlp
from src.preprocess import clean_series, fit_tokenizer, to_padded


def main(sample=None, epochs=None):
    if epochs:
        config.EPOCHS = epochs
    np.random.seed(config.SEED)
    tf.random.set_seed(config.SEED)

    train, val, test = load_data()
    if sample:
        train, val, test = (d.sample(min(sample, len(d)), random_state=config.SEED)
                            .reset_index(drop=True) for d in (train, val, test))
    print(f"train={len(train)}  val={len(val)}  test={len(test)}")

    # 1) Prétraitement
    for d in (train, val, test):
        d["clean"] = clean_series(d["text"])

    # 2) EDA
    run_eda(train, train["clean"])

    y_tr, y_va, y_te = (d["label"].values for d in (train, val, test))
    results, preds, histories = [], {}, {}

    # 3) Baselines (TF-IDF + LogReg / SVM)
    baselines = train_baselines(train["clean"], y_tr)
    for name, pipe in baselines.items():
        y_pred = pipe.predict(test["clean"])
        results.append(evaluate(name, y_te, y_pred))
        preds[name] = y_pred
        save(pipe, name.replace(" ", "_").replace("+", "").replace("__", "_"))

    # 4) MLP sur TF-IDF
    tfidf = make_tfidf().fit(train["clean"])
    Xtr, Xva, Xte = (tfidf.transform(d["clean"]) for d in (train, val, test))
    mlp, h_mlp = train_mlp(Xtr, y_tr, Xva, y_va)
    y_pred = predict_proba(mlp, Xte).argmax(1)
    results.append(evaluate("MLP (TF-IDF)", y_te, y_pred))
    preds["MLP (TF-IDF)"], histories["MLP"] = y_pred, h_mlp
    mlp.save(config.MODELS_DIR / "mlp.keras")
    joblib.dump(tfidf, config.MODELS_DIR / "tfidf_mlp.joblib")

    # 5) CNN (Kim)
    tok = fit_tokenizer(train["clean"])
    Ptr, Pva, Pte = (to_padded(tok, d["clean"]) for d in (train, val, test))
    cnn, h_cnn = train_cnn(Ptr, y_tr, Pva, y_va)
    y_pred = cnn.predict(Pte, verbose=0).argmax(1)
    results.append(evaluate("CNN 1D (Kim)", y_te, y_pred))
    preds["CNN 1D (Kim)"], histories["CNN"] = y_pred, h_cnn
    cnn.save(config.MODELS_DIR / "cnn.keras")
    (config.MODELS_DIR / "tokenizer.json").write_text(tok.to_json(), encoding="utf-8")

    # 6) Comparaison
    save_comparison(results)
    plot_confusion_matrices(preds, y_te)
    plot_learning_curves(histories)
    (config.REPORT_DIR / "results.json").write_text(
        json.dumps(results, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
    print("\nTerminé. Résultats dans reports/ — modèles dans models/")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--sample", type=int, default=None,
                    help="Sous-échantillonne chaque split (test rapide)")
    ap.add_argument("--epochs", type=int, default=None, help="Nombre max d'époques")
    args = ap.parse_args()
    main(args.sample, args.epochs)

In [ ]:
%%writefile /content/sentiment-multimodel/ui.py
"""Interface Gradio « Sentiment Lab » — pensée pour Google Colab (fonctionne aussi en local).

Onglets : 1) Données & EDA  2) Entraînement en direct  3) Résultats  4) Démo live  5) À propos
Lancer :  python ui.py        ou, dans Colab :  from ui import build; build().queue().launch(share=True)
"""
import matplotlib

matplotlib.use("Agg")

import os  # noqa: E402
import subprocess  # noqa: E402
import sys  # noqa: E402
from collections import Counter  # noqa: E402

import gradio as gr  # noqa: E402
import pandas as pd  # noqa: E402

from src import config  # noqa: E402

SRC_HF = "🤗 Dataset tweet_eval (anglais, 3 classes)"
SRC_CSV = "📄 Mon fichier CSV"
EMOJI = {"négatif": "😡", "neutre": "😐", "positif": "😊"}
COLOR = {"négatif": "#ef4444", "neutre": "#64748b", "positif": "#22c55e"}

# --------------------------------------------------------------------------- style
THEME = gr.themes.Soft(
    primary_hue="indigo", secondary_hue="violet", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
    radius_size="lg",
)

CSS = """
.gradio-container { max-width: 1180px !important; margin: auto; }
.hero { background: linear-gradient(120deg,#4f46e5 0%,#7c3aed 55%,#db2777 100%);
        border-radius: 20px; padding: 30px 34px; color: #fff; margin-bottom: 14px;
        box-shadow: 0 10px 30px rgba(79,70,229,.35); }
.hero h1 { margin: 0 0 6px 0; font-size: 2.1rem; font-weight: 800; color:#fff; }
.hero p  { margin: 0 0 14px 0; opacity: .92; font-size: 1.02rem; color:#fff; }
.pill { display:inline-block; background: rgba(255,255,255,.18); border:1px solid rgba(255,255,255,.35);
        padding: 4px 12px; border-radius: 999px; margin: 0 6px 6px 0; font-size:.82rem; color:#fff; }
.step-title { font-size: 1.05rem; font-weight: 700; margin: 4px 0 2px 0; }
.stepper { display:flex; gap:8px; flex-wrap:wrap; margin: 6px 0 2px 0; }
.chip { padding:6px 14px; border-radius:999px; font-size:.85rem; font-weight:600;
        background: var(--neutral-200); color: var(--neutral-600); }
.chip.done { background:#dcfce7; color:#166534; }
.chip.now  { background:#e0e7ff; color:#3730a3; animation: pulse 1.2s infinite; }
@keyframes pulse { 50% { opacity:.55; } }
.banner { border-radius:14px; padding:14px 18px; font-size:1.02rem; border:1px solid var(--border-color-primary); }
.banner.ok   { background:#ecfdf5; border-color:#86efac; color:#14532d; }
.banner.warn { background:#fffbeb; border-color:#fcd34d; color:#78350f; }
.banner.err  { background:#fef2f2; border-color:#fca5a5; color:#7f1d1d; }
.verdict { border-radius:18px; padding:20px 24px; color:#fff; text-align:center; }
.verdict .big { font-size:2.2rem; font-weight:800; }
.verdict .sub { opacity:.92; margin-top:4px; }
.banner.ok,   .banner.ok *   { color:#14532d !important; }
.banner.warn, .banner.warn * { color:#78350f !important; }
.banner.err,  .banner.err *  { color:#7f1d1d !important; }
.verdict, .verdict * { color: #fff !important; }
.hero, .hero * { color: #fff !important; }
.tabs > .tab-nav button { font-weight:600; }
button.primary { box-shadow: 0 4px 14px rgba(79,70,229,.35); }
footer { visibility: hidden; }
"""


# --------------------------------------------------------------------------- helpers
def _banner(text, kind="ok"):
    return f'<div class="banner {kind}">{text}</div>'


def _img(name):
    p = config.FIG_DIR / f"{name}.png"
    return str(p) if p.exists() else None


def _prepare_source(source, csv_path):
    """Applique le choix de source : copie le CSV utilisateur ou supprime l'ancien."""
    if source == SRC_CSV:
        if not csv_path:
            raise gr.Error("Importe d'abord un fichier CSV (colonnes `text` et `label`).")
        df = pd.read_csv(csv_path)
        if not {"text", "label"} <= set(df.columns):
            raise gr.Error("Le CSV doit contenir les colonnes `text` et `label`.")
        if not set(df["label"].unique()) <= {0, 1, 2}:
            raise gr.Error("`label` doit valoir 0 (négatif), 1 (neutre) ou 2 (positif).")
        df.to_csv(config.LOCAL_CSV, index=False)
    elif config.LOCAL_CSV.exists():
        config.LOCAL_CSV.unlink()


# --------------------------------------------------------------------------- 1) EDA
def explore(source, csv_path, sample):
    _prepare_source(source, csv_path)
    from src.data import load_data
    from src.eda import run_eda
    from src.preprocess import clean_series

    train, _, _ = load_data()
    if sample:
        train = train.sample(min(int(sample), len(train)), random_state=config.SEED)
    train = train.reset_index(drop=True)
    train["clean"] = clean_series(train["text"])
    run_eda(train, train["clean"])

    df = train.copy()
    df["classe"] = df["label"].map(dict(enumerate(config.LABELS)))
    df["n_mots"] = df["text"].str.split().str.len()
    summary = df.groupby("classe").agg(textes=("text", "size"),
                                       mots_moyens=("n_mots", "mean")).round(1).reset_index()
    summary["part_%"] = (summary["textes"] / summary["textes"].sum() * 100).round(1)
    examples = df[["classe", "text"]].sample(min(8, len(df)), random_state=1)
    top = Counter(" ".join(train["clean"]).split()).most_common(1)
    msg = _banner(f"✅ <b>{len(train):,}</b> textes analysés — mot le plus fréquent : "
                  f"<b>{top[0][0] if top else '—'}</b>. Les graphiques sont ci-dessous.")
    return (_img("eda_class_distribution"), _img("eda_text_length"), _img("eda_top_words"),
            summary, examples, msg)


# --------------------------------------------------------------------------- 2) Training
STEPS = ["Données", "EDA", "Baselines", "MLP", "CNN", "Rapport"]
MARKERS = [("train=", 1), ("EDA terminée", 2), ("=== TF-IDF + LinearSVM", 3),
           ("=== MLP", 4), ("=== CNN", 5), ("Terminé", 6)]
_CACHE = {}


def _stepper(done, failed=False):
    chips = []
    for i, name in enumerate(STEPS):
        cls = "done" if i < done else ("now" if i == done and not failed else "")
        chips.append(f'<span class="chip {cls}">{"✓ " if i < done else ""}{name}</span>')
    return f'<div class="stepper">{"".join(chips)}</div>'


def train(source, csv_path, sample, epochs):
    _prepare_source(source, csv_path)
    cmd = [sys.executable, "-u", "train_all.py", "--epochs", str(int(epochs))]
    if sample:
        cmd += ["--sample", str(int(sample))]
    env = {**os.environ, "TF_CPP_MIN_LOG_LEVEL": "3", "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, cwd=config.ROOT, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    log, done = [], 0
    yield "Démarrage…", _stepper(0)
    for line in proc.stdout:
        line = line.rstrip()
        if not line or line.startswith(("I0000", "W0000", "E0000", "WARNING:")):
            continue
        log.append(line)
        for marker, n in MARKERS:
            if marker in line:
                done = max(done, n)
        yield "\n".join(log[-250:]), _stepper(done)
    proc.wait()
    _CACHE.clear()
    if proc.returncode == 0:
        yield "\n".join(log[-250:]), _stepper(6) + _banner(
            "🎉 Entraînement terminé ! Va dans <b>Résultats</b> puis <b>Démo live</b>.")
    else:
        yield "\n".join(log[-250:]), _stepper(done, failed=True) + _banner(
            "❌ L'entraînement a échoué — regarde la fin du journal ci-dessus.", "err")


# --------------------------------------------------------------------------- 3) Results
def load_results():
    path = config.REPORT_DIR / "results.csv"
    if not path.exists():
        return (_banner("Aucun résultat pour l'instant — lance d'abord l'entraînement.", "warn"),
                None, None, None, None)
    df = pd.read_csv(path).round(4)
    best = df.loc[df["f1_macro"].idxmax()]
    base = df[df["modèle"].str.contains("LogReg")]["f1_macro"]
    gain = f" (<b>{(best['f1_macro'] - base.iloc[0]) * 100:+.1f} pts</b> de F1 vs baseline LogReg)" \
        if len(base) else ""
    banner = _banner(f"🏆 Meilleur modèle : <b>{best['modèle']}</b> — F1 macro "
                     f"<b>{best['f1_macro']:.3f}</b>, accuracy <b>{best['accuracy']:.3f}</b>{gain}")
    return (banner, df, _img("model_comparison"), _img("confusion_matrices"),
            _img("learning_curves"))


# --------------------------------------------------------------------------- 4) Live demo
def _models():
    if _CACHE:
        return _CACHE
    needed = ["cnn.keras", "tokenizer.json", "TF-IDF_LogReg.joblib", "mlp.keras",
              "tfidf_mlp.joblib"]
    if any(not (config.MODELS_DIR / f).exists() for f in needed):
        return None
    import joblib
    from tensorflow import keras
    from tensorflow.keras.preprocessing.text import tokenizer_from_json

    _CACHE.update(
        cnn=keras.models.load_model(config.MODELS_DIR / "cnn.keras"),
        mlp=keras.models.load_model(config.MODELS_DIR / "mlp.keras"),
        tok=tokenizer_from_json((config.MODELS_DIR / "tokenizer.json").read_text(encoding="utf-8")),
        tfidf=joblib.load(config.MODELS_DIR / "tfidf_mlp.joblib"),
        lr=joblib.load(config.MODELS_DIR / "TF-IDF_LogReg.joblib"),
    )
    return _CACHE


def predict(text):
    m = _models()
    if m is None:
        raise gr.Error("Aucun modèle entraîné : va d'abord dans l'onglet « Entraînement ».")
    if not text or not text.strip():
        raise gr.Error("Saisis un texte à analyser.")
    from src.preprocess import clean_text, to_padded

    clean = clean_text(text)
    probs = {
        "CNN 1D (Kim)": m["cnn"].predict(to_padded(m["tok"], [clean]), verbose=0)[0],
        "MLP (TF-IDF)": m["mlp"].predict(
            m["tfidf"].transform([clean]).toarray().astype("float32"), verbose=0)[0],
        "TF-IDF + LogReg": m["lr"].predict_proba([clean])[0],
    }
    votes = [config.LABELS[int(p.argmax())] for p in probs.values()]
    top, n = Counter(votes).most_common(1)[0]
    mean_conf = sum(p[config.LABELS.index(top)] for p in probs.values()) / len(probs)
    sub = ("Les 3 modèles sont d'accord" if n == 3
           else f"{n} modèles sur 3 — il y a un désaccord, le texte est ambigu")
    verdict = (f'<div class="verdict" style="background:{COLOR[top]}">'
               f'<div class="big">{EMOJI[top]} {top.capitalize()}</div>'
               f'<div class="sub">{sub} · confiance moyenne {mean_conf:.0%}</div></div>')
    out = [{l: float(x) for l, x in zip(config.LABELS, p)} for p in probs.values()]
    return (verdict, *out)


EXAMPLES = [
    ["I absolutely love this phone, the battery life is amazing!"],
    ["Worst customer service ever. I will never order here again."],
    ["The meeting is scheduled for Monday at 10am in room 4."],
    ["Not bad at all, but I expected a bit more for the price."],
    ["Can't believe how disappointing this was. Total waste of money."],
]

ABOUT = """
### 🎯 Objectif
Classer automatiquement des textes (tweets, avis…) en **négatif / neutre / positif** et
**comparer trois approches** avec le même protocole d'évaluation.

| Modèle | Entrée | Idée |
|---|---|---|
| **TF-IDF + LogReg / SVM** | n-grammes pondérés | Baseline rigoureuse à battre |
| **MLP** | vecteurs TF-IDF | Réseau dense : Dense → Dropout → Softmax |
| **CNN 1D (Kim 2014)** | embeddings appris | Conv1D k=3,4,5 → GlobalMaxPooling → Dense |

### 🧪 Rigueur méthodologique
- Négations conservées dans les stopwords (« not good » ≠ « good »)
- Vocabulaire et TF-IDF ajustés **uniquement sur le train** (pas de fuite de données)
- Early stopping sur la loss de validation, évaluation finale sur le **test**
- Métriques **macro** (précision, rappel, F1) + matrices de confusion normalisées

### 🛠️ Stack
Python · TensorFlow/Keras · scikit-learn · NLTK · pandas · matplotlib/seaborn · Gradio
"""


# --------------------------------------------------------------------------- app
def build() -> gr.Blocks:
    with gr.Blocks(theme=THEME, css=CSS, title="Sentiment Lab") as demo:
        gr.HTML(
            '<div class="hero"><h1>💬 Sentiment Lab</h1>'
            "<p>Analyse de sentiments multi-modèles : de la baseline TF-IDF au CNN 1D, "
            "comparés sur les mêmes données.</p>"
            '<span class="pill">TF-IDF + LogReg</span><span class="pill">MLP</span>'
            '<span class="pill">CNN 1D · Kim</span><span class="pill">NLP · Keras</span></div>'
        )

        with gr.Accordion("⚙️ Paramètres (communs aux onglets Données et Entraînement)", open=True):
            with gr.Row():
                source = gr.Radio([SRC_HF, SRC_CSV], value=SRC_HF, label="Source des données")
                csv_file = gr.File(label="CSV (colonnes `text`, `label` : 0/1/2)",
                                   file_types=[".csv"], type="filepath")
            with gr.Row():
                sample = gr.Slider(0, 50000, value=5000, step=500,
                                   label="Nombre de textes par split (0 = tout le dataset)")
                epochs = gr.Slider(2, 30, value=10, step=1, label="Époques max (early stopping actif)")

        with gr.Tabs():
            # ---- 1) Données & EDA
            with gr.Tab("📊 1 · Données & EDA"):
                eda_btn = gr.Button("Analyser les données", variant="primary")
                eda_msg = gr.HTML()
                with gr.Row():
                    eda_img1 = gr.Image(label="Distribution des classes", interactive=False,
                                        type="filepath")
                    eda_img2 = gr.Image(label="Longueur des textes", interactive=False,
                                        type="filepath")
                eda_img3 = gr.Image(label="Mots les plus fréquents par classe",
                                    interactive=False, type="filepath")
                with gr.Row():
                    eda_summary = gr.Dataframe(label="Résumé par classe", interactive=False)
                    eda_examples = gr.Dataframe(label="Exemples de textes", interactive=False,
                                                wrap=True)
                eda_btn.click(explore, [source, csv_file, sample],
                              [eda_img1, eda_img2, eda_img3, eda_summary, eda_examples, eda_msg])

            # ---- 2) Entraînement
            with gr.Tab("🏋️ 2 · Entraînement"):
                gr.Markdown("Entraîne les **4 modèles** (LogReg, SVM, MLP, CNN) d'un seul clic. "
                            "Astuce : commence avec 5 000 textes pour un test rapide.")
                train_btn = gr.Button("🚀 Lancer l'entraînement", variant="primary")
                stepper = gr.HTML(_stepper(0))
                logs = gr.Textbox(label="Journal d'entraînement", lines=16, max_lines=16,
                                  interactive=False, autoscroll=True)
                train_btn.click(train, [source, csv_file, sample, epochs], [logs, stepper])

            # ---- 3) Résultats
            with gr.Tab("📈 3 · Résultats"):
                res_btn = gr.Button("Afficher / rafraîchir les résultats", variant="primary")
                res_banner = gr.HTML()
                res_table = gr.Dataframe(label="Métriques sur le jeu de test", interactive=False)
                res_bar = gr.Image(label="Comparaison des modèles", interactive=False,
                                   type="filepath")
                with gr.Row():
                    res_cm = gr.Image(label="Matrices de confusion", interactive=False,
                                      type="filepath")
                    res_lc = gr.Image(label="Courbes d'apprentissage", interactive=False,
                                      type="filepath")
                res_btn.click(load_results, None, [res_banner, res_table, res_bar, res_cm, res_lc])

            # ---- 4) Démo live
            with gr.Tab("🔮 4 · Démo live"):
                gr.Markdown("Tape un texte **en anglais** (modèles entraînés sur tweet_eval) "
                            "et compare la réponse des 3 modèles.")
                txt = gr.Textbox(label="Ton texte", lines=3,
                                 placeholder="This product is fantastic, I love it!")
                go = gr.Button("Analyser le sentiment", variant="primary")
                verdict = gr.HTML()
                with gr.Row():
                    l_cnn = gr.Label(label="CNN 1D (Kim)", num_top_classes=3)
                    l_mlp = gr.Label(label="MLP (TF-IDF)", num_top_classes=3)
                    l_lr = gr.Label(label="TF-IDF + LogReg", num_top_classes=3)
                gr.Examples(EXAMPLES, inputs=txt, label="Exemples")
                outs = [verdict, l_cnn, l_mlp, l_lr]
                go.click(predict, txt, outs)
                txt.submit(predict, txt, outs)

            # ---- 5) À propos
            with gr.Tab("ℹ️ À propos"):
                gr.Markdown(ABOUT)
    return demo


if __name__ == "__main__":
    build().queue().launch()

## 3️⃣ Lancer l'interface 🚀

In [ ]:
%cd /content/sentiment-multimodel
import sys, os
sys.path.insert(0, os.getcwd())

import gradio as gr
gr.close_all()                      # ferme une éventuelle instance précédente

from ui import build
demo = build()
demo.queue().launch(share=True, debug=False)